In [1]:

import re

# Regex for detecting earnings + currency
EARN_PAT = re.compile(r'([\d\.\s,]{4,})\s*(SEK|EUR|\€|NOK|USD|\$|KR)\b', re.IGNORECASE)

# Currency conversion rates to SEK
CURRENCY_RATE = {
    "SEK": 1.0,
    "NOK": 1.0,
    "EUR": 11.0,
    "USD": 10.0,
    "$": 10.0,
    "€":  11.0,
    "KR": 1.0,
    "AUD": 6.5,
    "DKK": 1.5
}


# Solvalla Catalog — Reengineered Parser\n

In [4]:

from __future__ import annotations
from pathlib import Path
import re, pdfplumber, pandas as pd

PDF_CANDIDATES = [
    Path.cwd() / "SOLVALLA_YS_25.pdf",
    Path("/mnt/data/SOLVALLA_YS_25.pdf"),
    Path(r"C:\\Users\\RunarSkjaerpe\\Kataloger\\SOLVALLA_YS_25.pdf"),
]
pdf_path = next((p for p in PDF_CANDIDATES if p.exists()), None)
assert pdf_path is not None, "Place 'SOLVALLA_YS_25.pdf' next to this notebook or update PDF_CANDIDATES."

OUT_DIR = Path.cwd()
OUT_V4_XLSX = OUT_DIR / "SOLVALLA_YS_25_REENG.xlsx"
OUT_V4_CSV  = OUT_DIR / "SOLVALLA_YS25_catalog_FINAL_FIXED_v4_REENG.csv"
OUT_V6_XLSX = OUT_DIR / "SOLVALLA_YS25_FINAL_FIXED_v6_REENG.xlsx"
OUT_V6B_XLSX = OUT_DIR / "SOLVALLA_YS25_FINAL_FIXED_v6b_REENG.xlsx"
OUT_FINAL_ENRICH_XLSX = OUT_DIR / "SOLVALLA_YS25_FINAL_FIXED_v6c_DAMHYBRID2_STATS_REENG.xlsx"
ID_PAT    = re.compile(r"\b\d{2}-\d{3,5}\b")
SEX_PAT   = re.compile(r"\b(hingst|sto|vallack|valack)\b", re.IGNORECASE)
COLOR_PAT = re.compile(r"\b(svart|svartbrun|brun|mörkbrun|morkbrun|ljusbrun|fux|skimmel)\b", re.IGNORECASE)

def _normalize(s: str) -> str:
    return (s or "").replace("\u00a0"," ").replace("–","-")


In [5]:

# --- Helpers: French Blood & Inbreeding (supports both orders and spellings) ---
import re


# French Blood patterns
FR_BEFORE = re.compile(r"(?P<pct>\d+[.,]?\d*)\s*%\s*franskt\s*blod", re.IGNORECASE)
FR_AFTER  = re.compile(r"franskt\s*blod\s*[:\-]\s*(?P<pct>\d+[.,]?\d*)\s*%", re.IGNORECASE)  # Colon/dash required

# Inbreeding patterns  
INAV_KEY  = r"inav(?:e)?l(?:skoeff(?:icient)?)?"
IN_BEFORE = re.compile(rf"(?P<pct>\d+[.,]?\d*)\s*%\s*{INAV_KEY}", re.IGNORECASE)
IN_AFTER  = re.compile(rf"{INAV_KEY}\s*[:\-]\s*(?P<pct>\d+[.,]?\d*)\s*%", re.IGNORECASE)  # Colon/dash required

def _to_pct_float(s):
    if not s: return None
    s = str(s).replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return None


In [6]:

# --- Dam performance helpers ---
TIME_PAT_DAM   = re.compile(r"\b(?P<m>\d{2})[.,](?P<d>\d)\b")
EARN_PAT_DAM   = re.compile(r'(?P<num>[\d\.\s,]{4,})\s*(?P<cur>SEK|EUR|\€|NOK|USD|AUD|DKK|\$|KR)\b', re.IGNORECASE)
STARTS_PAT_DAM = re.compile(r"\b(?P<st>\d+)\s*st[: ]\s*(?P<w>\d+)\s*[–-]\s*(?P<p2>\d+)\s*[–-]\s*(?P<p3>\d+)\b", re.IGNORECASE)

CURRENCY_RATE = {'SEK':1.0,'KR':1.0,'EUR':11.0, '€': 11.0, 'AUD': 6.5, 'NOK':1.0, 'DKK': 1.5, 'USD':10.0,'$':10.0}


In [7]:
import re

def lower_heading_indices(lines):
    idxs = []
    for i, L in enumerate(lines):
        # Look for lines like 'CITRON  f. 07 ...' (dam / grand-dam headers)
        if re.search(r"\bf\.?\s*\d{2,4}", L) and L.strip() and L.strip().split()[0].isupper():
            idxs.append(i)
    return idxs

def header_name(line):
    # Extract the name before 'f. <year>'
    m = re.split(r"\bf\.?\s*\d{2,4}", line)
    name = (m[0] if m else line).strip()
    return name.title()

def section_end(lines, start_idx):
    if start_idx is None:
        return None
    idxs = lower_heading_indices(lines)
    next_idxs = [i for i in idxs if i > start_idx]
    return next_idxs[0] if next_idxs else len(lines)

def agg_offspring(lines, start_idx, end_idx, exclude_name=None):
    def _is_heading_line(L: str) -> bool:
        if not L: return False
        S = str(L).strip().replace("'", "'")  # Fixed: use regular quotes
        if not S: return False
        return bool(re.search(r"\bf\.?\s*\d{2,4}\b", S))

     # Normalize names: remove parentheses content and non-letters; lowercase
    def _norm_name(s):
        if not s:
            return ""
        s = re.sub(r"\((?![^)]*död)[^)]*\)", "", s, flags=re.IGNORECASE)
        s = s.replace("'", "'")  # Fixed: use regular quotes
        s = re.sub(r"[^A-Za-zÅÄÖåäöÉéØøÜüÆæÇçÑñ\s]", "", s)
        s = re.sub(r"\s+", " ", s).strip().lower()
        return s

    def _clean_doubled_chars(line):
        """
        Clean lines where MOST characters are doubled (e.g., 'HHeelllloo' -> 'Hello')
        This happens with highlighted text in some PDFs.
        Only clean if at least 50% of characters appear to be doubled.
        """
        if len(line) < 10:
            return line
        
        doubled_count = 0
        total_count = 0
        
        for i in range(len(line) - 1):
            if line[i] != ' ':
                total_count += 1
                if line[i] == line[i + 1]:
                    doubled_count += 1
        
        if total_count > 0 and doubled_count / total_count > 0.5:
            cleaned = ""
            i = 0
            while i < len(line):
                if i + 1 < len(line) and line[i] == line[i + 1]:
                    cleaned += line[i]
                    i += 2
                else:
                    cleaned += line[i]
                    i += 1
            return cleaned
        
        return line

    if start_idx is None or end_idx is None:
        return {'st':0,'w':0,'p2':0,'p3':0,'earn_raw':0,'earn_sek':0,
                'born_inc':0,'started':0,'total':0,'best_earn_sek':0}

    block = lines[start_idx+1:end_idx]
    cut = None
    for j, _L in enumerate(block):
        if _is_heading_line(_L):
            cut = j
            break
    eff_block = block[:cut] if cut is not None else block

    s = w = p2 = p3 = 0
    earn_raw_sum = earn_sek_sum = 0
    born_inc = 0
    born_le_2021 = 0
    started = 0
    total = 0
    best_earn_sek = 0

    for T_raw in eff_block:
        T_raw = T_raw.strip()
        if not T_raw:
            continue
        
        # CRITICAL: Clean doubled characters from highlighted text
        T = _clean_doubled_chars(T_raw)
        
        norm_ex = _norm_name(exclude_name)
        norm_line = _norm_name(T)
        if norm_ex and norm_ex in norm_line:
            continue


        is_sibling = re.match(
           r"^\s*\d+\s*[A-Za-zÅÄÖåäöÉéØøÜüÆæÇç][^\d\(]*(?:\([^)]*\))?\s*[svh]\s+e\s+",
           T, re.IGNORECASE
         ) is not None

        if not is_sibling:
            if re.search(r"\bfödd\b|\bborn\b", T, flags=re.IGNORECASE):
                born_inc += 1
            continue

        total += 1
        m_year = re.match(r"\s*(\d{2,4})", T)  # Removed \b
        if m_year:
            yy = int(m_year.group(1))
            if yy < 100:
                year_full = 2000 + yy if yy < 50 else 1900 + yy
            else:
                year_full = yy
            if year_full <= 2022:
                born_le_2021 += 1

        pm = re.search(r"(\d+)\s+(\d+)[–-](\d+)[–-](\d+)\b", T)
        if pm:
            s  += int(pm.group(1))
            w  += int(pm.group(2))
            p2 += int(pm.group(3))
            p3 += int(pm.group(4))
            started += 1

        amount = None
        currency = None

        # Pattern A: amount CUR starts - earnings use dots/spaces, NOT commas
        m_a = re.search(
            #r"(\d[\d\.\s]+)\s+(SEK|EUR|€|NOK|USD|\$|KR)\s+\d{1,3}\s+\d+[–-]\d+[–-]\d+\b",
            r"(\d[\d\.\s,]{2,})\s*(SEK|EUR|\€|€|NOK|DKK|AUD|USD|\$|KR)\s+\d{1,3}\s+\d+[–-]\d+[–-]\d+\b",
            T, re.IGNORECASE
        )
        # Pattern B: CUR amount starts
        m_b = re.search(
            #r"(SEK|EUR|€|NOK|USD|\$|KR)\s*(\d[\d\.\s]+)\s+\d{1,3}\s+\d+[–-]\d+[–-]\d+\b",
            r"(SEK|EUR|\€|€|NOK|DKK|AUD|USD|\$|KR)\s*(\d[\d\.\s,]{2,})\s+\d{1,3}\s+\d+[–-]\d+[–-]\d+\b",
            
            T, re.IGNORECASE
        )
        # Pattern C: No currency - must have dots/spaces to distinguish from times
        m_c = re.search(
            r"\s(\d[\d\.\s]+)\s+\d{1,3}\s+\d+[–-]\d+[–-]\d+\b",
            T
        )

        if m_a:
            amount = int(re.sub(r"[^\d]", "", m_a.group(1)))
            currency = m_a.group(2).upper()
        elif m_b:
            amount = int(re.sub(r"[^\d]", "", m_b.group(2)))
            currency = m_b.group(1).upper()
        elif m_c:
            amount = int(re.sub(r"[^\d]", "", m_c.group(1)))
            currency = "SEK"

        if amount is not None:
            earn_raw_sum += amount
            rate = CURRENCY_RATE.get(currency, 1.0)
            earn_sek = int(round(amount * rate))
            earn_sek_sum += earn_sek
            
            # Track best individual earnings
            if earn_sek > best_earn_sek:
                best_earn_sek = earn_sek

    return {'st': s, 'w': w, 'p2': p2, 'p3': p3,
            'earn_raw': earn_raw_sum, 'earn_sek': earn_sek_sum,
            'born_inc': born_le_2021, 'born_le_2021': born_le_2021,
            'started': started, 'total': total, 'best_earn_sek': best_earn_sek}





In [8]:

def parse_mother_perf(lines, dam_idx):
    """
    Return (best_time_display, earn_raw, currency, earn_sek, starts, w, p2, p3)
    by scanning the dam section only.
    """
    if dam_idx is None:
        return None, 0, None, 0, 0, 0, 0, 0

    # Find section end using helper
    end_idx = section_end(lines, dam_idx)
    block = lines[dam_idx:end_idx]

    # Defaults
    best_time = None
    earn_raw = 0
    currency = None
    earn_sek = 0
    starts = w = p2 = p3 = 0

    # Try to find a line with a dam performance summary
    for T in block:
        T = T.strip()
        if not T:
            continue

        # Best time like 13,0m or 10,9ak
        tm = re.search(r"\b(\d{1,2, '€': 11.0})[.,](\d)\s*[a-z]{0,2}\b", T, flags=re.IGNORECASE)
        if tm and best_time is None:
            best_time = f"{tm.group(1)},{tm.group(2)}"

        # Earnings with currency
        em = EARN_PAT_DAM.search(T) if 'EARN_PAT_DAM' in globals() else None
        if em and earn_raw == 0:
            digits = re.sub(r"[^\d]", "", em.group('num'))
            if digits:
                earn_raw = int(digits)
                currency = em.group('cur').upper()
                rate = CURRENCY_RATE.get(currency, 1.0) if 'CURRENCY_RATE' in globals() else 1.0
                earn_sek = int(round(earn_raw * rate))

        # Starts summary like '33 st: 13–4–1'
        sm = STARTS_PAT_DAM.search(T) if 'STARTS_PAT_DAM' in globals() else None
        if sm and starts == 0:
            starts = int(sm.group('st'))
            w = int(sm.group('w'))
            p2 = int(sm.group('p2'))
            p3 = int(sm.group('p3'))

    return best_time, earn_raw, currency, earn_sek, starts, w, p2, p3


In [9]:

def extract_page(pdf, page_num: int) -> dict:
    txt = _normalize(pdf.pages[page_num-1].extract_text() or "")
    lines = txt.splitlines()

    # Robust sire parser: find 'starts' and scan upward for earn/time/name
    def _parse_sire_block(lines, stop_idx):
        N = min(stop_idx or len(lines), len(lines))
        starts_re = re.compile(r"(\d+)\s*st[: ]\s*(\d+)[–-](\d+)[–-](\d+)\b", re.IGNORECASE)
        time_re   = re.compile(r"\b\d{1,2}\.\d{1,2},\d[a-z]{0,2}k?\b", re.IGNORECASE)
        earn_a    = re.compile(r"(\d[\d\.\s,]{2,})\s*(SEK|EUR|€|NOK|DKK|USD|\$|KR)\b", re.IGNORECASE)
        earn_b    = re.compile(r"(SEK|EUR|€|NOK|DKK|USD|\$|KR)\s*(\d[\d\.\s,]{2,})\b", re.IGNORECASE)
        def is_caps_name(s):
            s = s.strip(); return s and not re.search(r"\d", s) and s == s.upper()
        for i in range(0, N):
            m = starts_re.search(lines[i])
            if not m: continue
            st, w, p2, p3 = map(int, m.groups())
            name = time = None; earn = None; cur = None
            for j in range(i-1, max(-1, i-9), -1):
                L = lines[j].strip()
                if earn is None:
                    ma = earn_a.search(L); mb = earn_b.search(L)
                    if ma:
                        earn = int(re.sub(r"\D", "", ma.group(1))); cur = ma.group(2).upper()
                    elif mb:
                        earn = int(re.sub(r"\D", "", mb.group(2))); cur = mb.group(1).upper()
                    elif re.search(r"\d", L):
                        maybe = int(re.sub(r"\D", "", L))
                        if maybe >= 1000:
                            earn = maybe; cur = 'SEK'
                if time is None:
                    mt = time_re.search(L)
                    if mt: time = mt.group(0)
                if name is None and is_caps_name(L):
                    name = L.title()
                if name and time and earn is not None:
                    break
            if name and time and earn is not None:
                rate = 1.0
                try:
                    rate = CURRENCY_RATE.get(cur, 1.0)
                except Exception:
                    pass
                sek = int(round(earn * rate))
                return name, time, earn, cur, sek, st, w, p2, p3
        return (None,)*9

    header_f_pat = re.compile(r"\bf\.?\s*\d{2,4}\b", re.IGNORECASE)

    # Page-level badge: 'Eligible for American Stakes races'
    us_stakes_eligible = 1 if re.search(r"Eligible\s+for\s+American\s+Stakes\s+races", txt, re.IGNORECASE) else 0

    # Parse French Blood % and Inbreeding % (both orders & spellings)
    _fm = FR_AFTER.search(txt) or FR_BEFORE.search(txt)
    _im = IN_AFTER.search(txt) or IN_BEFORE.search(txt)
    french_pct = _to_pct_float(_fm.group('pct')) if _fm else None
    inavl_pct  = _to_pct_float(_im.group('pct')) if _im else None

    # Horse name

    name = None

    # Step 1: Find the line with "f. <date>"
    birth_idx = None
    for i, L in enumerate(lines[:60]):
        if re.search(r"\bf\.\s*\d{1,2}\s+[A-Za-zÅÄÖåäöÉéØøÜüÆæ]+\s+\d{4}", L, re.IGNORECASE):
            birth_idx = i
            break
    
    # Step 2: Search upward for the most recent uppercase name line
    if birth_idx is not None:
        for k in range(birth_idx - 1, max(birth_idx - 8, -1), -1):
            cand = lines[k].strip()
            if cand and 3 < len(cand) <= 40 and cand.upper() == cand and not re.search(r"\d", cand):
                name = cand.title()
                break
    
    # Step 3: Fallback to "presenterar" method if still not found
    if name is None:
        for i, L in enumerate(lines[:25]):
            if 'presenterar' in L.lower():
                for j in range(i + 1, min(i + 10, len(lines))):
                    cand = lines[j].strip()
                    if cand and 3 < len(cand) <= 40 and cand.upper() == cand and not re.search(r"\d", cand):
                        name = cand.title()
                        break
                break


    # Top-of-page stats
    # Match registration ID + color + sex + birth date
    m = re.search(
        r"((?:[A-Z]{1,3}-?\d{2,}-?\d{3,6}))\s+([A-Za-zÅÄÖåäöÉéØøÜüÆæÇç\-]+)\s+"
        r"(hingst|sto|vallack|valack)\s+f\.?\s*(\d{1,2}\s+[A-Za-zÅÄÖåäöÉéØøÜüÆæ]+\s+\d{4})",
        txt,
        re.IGNORECASE
    )
    #m = re.search(r"(\d{2}-\d{3,5})\s+([A-Za-zÅÄÖåäöÉéØøÜüÆæÇç\-]+)\s+(hingst|sto|vallack|valack)\s+f\.?\s*(\d{2,4})", txt, re.IGNORECASE)
    id_val = m.group(1) if m else None
    color  = m.group(2).title().replace('Morkbrun','Mörkbrun') if m else None
    sex    = m.group(3).title() if m else None

 

    # Birth date like 'f. 8 april 2023'
    birth_date_str = None
    md = re.search(r"\bf\.?\s*(\d{1,2})\s+([A-Za-zÅÄÖåäöÉéØøÜÆæçñ]+)\s+(\d{4})", txt, re.IGNORECASE)
    if md:
        birth_date_str = f"{int(md.group(1))} {md.group(2)} {md.group(3)}"

    hdrs = lower_heading_indices(lines)
    dam_idx = hdrs[0] if len(hdrs)>0 else None
    damdam_idx = hdrs[1] if len(hdrs)>1 else None
    ggdam_idx  = hdrs[2] if len(hdrs)>2 else None

    dam_name = header_name(lines[dam_idx]) if dam_idx is not None else None
    damdam_name = header_name(lines[damdam_idx]) if damdam_idx is not None else None
    ggdam_name  = header_name(lines[ggdam_idx]) if ggdam_idx is not None else None

    # Compute section ends based on the next header line ('f.' + year)
    def next_header_after(idx):
        if idx is None:
            return len(lines)
        for j in range(idx+1, len(lines)):
            if header_f_pat.search(lines[j]):
                return j
        return len(lines)

    dam_end    = next_header_after(dam_idx)
    damdam_end = next_header_after(damdam_idx)
    ggdam_end  = next_header_after(ggdam_idx)

    # Parse sire block up to the dam header
    sire_name, sire_best, sire_earn_raw, sire_cur, sire_earn_sek, sire_st, sire_w, sire_p2, sire_p3 = _parse_sire_block(lines, dam_idx)


    # Dam Birth Year and BLUP (from dam header line and nearby text)
    dam_birth_year = None
    dam_blup = None
    if dam_idx is not None:
        _dam_line = lines[dam_idx]
        mby = re.search(r"\bf\.?\s*(\d{2,4})", _dam_line, re.IGNORECASE)
        if mby:
            y = int(mby.group(1))
            if y < 100:
                dam_birth_year = (2000 + y) if y < 50 else (1900 + y)
            else:
                dam_birth_year = y
        # Look for BLUP value on the dam header line and maybe the next few lines
        nearby = "\n".join(lines[dam_idx:dam_idx+3])
        mb = re.search(r"BLUP[^\d]{0,30}(\d{2,3})", nearby, re.IGNORECASE)
        if mb:
            dam_blup = int(mb.group(1))


            
    sibs = agg_offspring(lines, dam_idx, dam_end)
    gdam = agg_offspring(lines, damdam_idx, damdam_end, exclude_name=dam_name)
    ggdm = agg_offspring(lines, ggdam_idx, ggdam_end)

    m_best, m_earn, m_cur, m_sek, m_starts, m_w, m_p2, m_p3 = parse_mother_perf(lines, dam_idx)

    return {
        "Page": page_num,
        "Name": name,
        "ID": id_val,
        "Sex": sex,
        "Color": color,
        "French Blood % (display)": french_pct,
        "Birth Date": birth_date_str,
        "Inbreeding % (display)": inavl_pct,
        "Dam": dam_name,
        "Sire": sire_name,
        "Sire Best Time": sire_best,
        "Sire Earnings": sire_earn_raw,
        "Sire Currency": sire_cur,
        "Sire Earnings (SEK)": sire_earn_sek,
        "Sire Starts": sire_st,
        "Sire 1st": sire_w,
        "Sire 2nd": sire_p2,
        "Sire 3rd": sire_p3,
        "Dam's Dam": damdam_name,
        "Granddam's Dam": ggdam_name,
        "Dam Best Time": m_best,
        "Dam Earnings": m_earn,
        "Dam Currency": m_cur,
        "Dam Earnings (SEK)": m_sek,
        "Dam Starts": m_starts,
        "Dam 1st": m_w,
        "Dam 2nd": m_p2,
        "Dam 3rd": m_p3,
        "Sibling Earnings": sibs["earn_raw"],
        "Sibling Earnings (SEK)": sibs["earn_sek"],
        "Sibling Starts": sibs["st"],
        "Sibling 1st": sibs["w"],
        "Sibling 2nd": sibs["p2"],
        "Sibling 3rd": sibs["p3"],
        "Siblings Started": sibs["started"],
        "Total Siblings": sibs["total"],
        "Siblings Born ≤ 2021 (excl 2022–24)": sibs["born_inc"],
        "Gdam Offspring Earnings (excl Dam)": gdam["earn_raw"],
        "Gdam Offspring Earnings (SEK, excl Dam)": gdam["earn_sek"],
        "Gdam Offspring Starts (excl Dam)": gdam["st"],
        "Gdam Offspring 1st (excl Dam)": gdam["w"],
        "Gdam Offspring 2nd (excl Dam)": gdam["p2"],
        "Gdam Offspring 3rd (excl Dam)": gdam["p3"],
        "Gdam Offspring Started (excl Dam)": gdam["started"],
        "Gdam Total Offspring (excl Dam)": gdam["total"],
        "Gdam Offspring Born ≤ 2021 (excl 2022–24)": gdam.get("born_le_2021", gdam.get("born_inc", 0)),
        "Ggdam Offspring Earnings (incl Gdam)": ggdm["earn_raw"],
        "Ggdam Offspring Earnings (SEK, incl Gdam)": ggdm["earn_sek"],
        "Ggdam Offspring Starts (incl Gdam)": ggdm["st"],
        "Ggdam Offspring 1st (incl Gdam)": ggdm["w"],
        "Ggdam Offspring 2nd (incl Gdam)": ggdm["p2"],
        "Ggdam Offspring 3rd (incl Gdam)": ggdm["p3"],
        "Ggdam Offspring Started (incl Gdam)": ggdm["started"],
        "Ggdam Total Offspring (incl Gdam)": ggdm["total"],
        "Ggdam Offspring Born ≤ 2021 (excl 2022–24)": ggdm.get("born_le_2021", ggdm.get("born_inc", 0)),
        "Dam Birth Year": dam_birth_year,
        "Dam BLUP": dam_blup,
        "Eligible for American Stakes races": us_stakes_eligible
    }


In [10]:

def build_dataframe(pdf_path: Path, start_page: int | None=None, end_page: int | None=None) -> pd.DataFrame:
    rows = []
    with pdfplumber.open(str(pdf_path)) as pdf:
        a = start_page or 1
        b = end_page or len(pdf.pages)
        for p in range(a, b+1):
            try:
                rows.append(extract_page(pdf, p))
            except Exception as e:
                rows.append({"Page": p})
                print(f"[WARN] Page {p}: {e}")
    return pd.DataFrame(rows).sort_values("Page")

df = build_dataframe(pdf_path, start_page=25, end_page=154)


print("Built df:", df.shape)


Built df: (130, 58)


In [11]:

# --- Repair ID / Sex / Color from the top header for any rows where they are missing ---
import re, pdfplumber

def _norm_header(s: str) -> str:
    return (s or "").replace("\u00a0", " ").replace("–", "-")

COLOR_WORDS = [
    "svartbrun","mörkbrun","morkbrun","ljusbrun","mörkfux","morkfux",
    "svart","brun","fux","skimmel"
]
SEX_WORDS = ["hingst","sto","vallack","valack"]

COL_RE = r"(?P<color>" + "|".join(COLOR_WORDS) + r")"
SEX_RE = r"(?P<sex>" + "|".join(SEX_WORDS) + r")"
ID_RE  = r"(?P<id>\d{2}[-–]\d{3,5})"

_PATTERNS = [
    re.compile(ID_RE + r"\s+" + COL_RE + r"\s+" + SEX_RE + r"\b", re.IGNORECASE),
    re.compile(ID_RE + r"\s+" + SEX_RE + r"\s+" + COL_RE + r"\b", re.IGNORECASE),
]

def parse_id_sex_color_from_header(text: str):
    t = _norm_header(text)
    first_lines = t.splitlines()[:12]
    block = re.sub(r"\s+", " ", " ".join(first_lines)).strip()

    for pat in _PATTERNS:
        m = pat.search(block)
        if m:
            id_val = m.group("id")
            sex    = m.group("sex").title()
            color  = m.group("color").title().replace("Morkbrun", "Mörkbrun").replace("Morkfux","Mörkfux")
            if color.lower() == "morkbrun": color = "Mörkbrun"
            if color.lower() == "morkfux":  color = "Mörkfux"
            return id_val, sex, color

    id_m = re.search(ID_RE, block)
    sex_m = re.search(SEX_RE, block, re.IGNORECASE)
    col_m = re.search(COL_RE, block, re.IGNORECASE)

    id_val = id_m.group("id") if id_m else None
    sex    = sex_m.group("sex").title() if sex_m else None
    color  = (col_m.group("color").title() if col_m else None)
    if color:
        color = color.replace("Morkbrun", "Mörkbrun").replace("Morkfux","Mörkfux")
    return id_val, sex, color

assert 'df' in globals() and 'pdf_path' in globals(), "Make sure df and pdf_path exist"

need_fix = df[(df['ID'].isna()) | (df['Sex'].isna()) | (df['Color'].isna())]['Page'].astype(int).tolist()

fixed = 0
with pdfplumber.open(str(pdf_path)) as pdf:
    for p in need_fix:
        try:
            txt = pdf.pages[p-1].extract_text() or ""
        except Exception:
            continue
        id_val, sex, color = parse_id_sex_color_from_header(txt)
        mask = (df['Page'] == p)
        if id_val: df.loc[mask, 'ID'] = id_val
        if sex:    df.loc[mask, 'Sex'] = sex
        if color:  df.loc[mask, 'Color'] = color
        if id_val or sex or color:
            fixed += 1

print(f"Repaired ID/Sex/Color on {fixed} pages (out of {len(need_fix)} needing fixes).")


Repaired ID/Sex/Color on 130 pages (out of 130 needing fixes).


In [19]:
# --- Dam performance: choose a time ONLY if earnings/starts occur within next 3 lines ---
import re, pdfplumber

CURRENCY_RATE = globals().get(
    "CURRENCY_RATE",
    {'SEK':1.0,'KR':1.0,'EUR':11.0, '€': 11.0, 'AUD': 6.5, 'NOK':1.0, 'DKK': 1.5, 'USD':10.0,'$':10.0}
)

def _norm_text(s: str) -> str:
    return (s or "").replace("\u00a0"," ").replace("–","-")

def _strip_country(name: str) -> str:
    return re.sub(r"\([^)]*\)", "", name or "").strip()

def _words(name: str):
    n = _strip_country(name).replace("'","'").replace("`","'")
    return [t.lower() for t in re.findall(r"[0-9A-Za-zÅÄÖåäöÆæØøÇç]+", n, flags=re.UNICODE)]

def _sig_prefixes(name: str):
    out = []
    for w in _words(name):
        out.append(w[:3] if len(w) >= 3 else w)
    return out

def _line_tokens(line: str):
    return [t.lower() for t in re.findall(r"[0-9A-Za-zÅÄÖåäöÆæØøÇç]+", line or "", flags=re.UNICODE)]

def _line_has_prefixes(line: str, prefixes: list[str]) -> int:
    ltoks = _line_tokens(line)
    return sum(any(tok.startswith(p) for tok in ltoks) for p in prefixes)

#def _clean_int(s):
#    return int(re.sub(r"[^0-9]","", s)) if s else None

def _clean_int(s):
    if not s:
        return None
    cleaned = re.sub(r"[^0-9]", "", s)
    return int(cleaned) if cleaned else None

def parse_dam_own_performance(dam_name: str, lines: list[str], dam_header_idx: int, debug: bool = False):
    """
    Find the dam's own performance by searching BEFORE the dam header line.
    The dam appears in the pedigree chart: Name on one line, then TIME, EARNINGS, STARTS on next few lines
    """
    if dam_header_idx is None or dam_header_idx < 5:
        return (None,)*8
    
    # Search backwards from dam header (up to 30 lines before, but not before line 8)
    search_start = max(8, dam_header_idx - 30)
    search_section = lines[search_start:dam_header_idx]
    
    dam_words = _words(dam_name)
    if not dam_words:
        return (None,)*8
    
    # First, try to find the dam name line
    for i in range(len(search_section) - 1, -1, -1):
        line = search_section[i].strip()
        if not line:
            continue
        
        # Skip lines that are section headers (have "f. <year>")
        if re.search(r"\bf\.?\s*\d{2,4}\b", line, re.IGNORECASE):
            continue
        
        # Check if line contains dam name tokens
        line_tokens = _line_tokens(line)
        matches = sum(1 for dw in dam_words if any(tok.startswith(dw[:3]) for tok in line_tokens))
        
        # Require better matching: 
        # - If dam has 2+ words, need at least 2 matches
        # - If dam has 1 word, need 1 match
        if len(dam_words) >= 2 and matches < 2:
            continue
        elif matches == 0:
            continue
        
        # Skip if line starts with starts pattern (like "33 st: 15-6-2 ELATION RIDE")
        # This means it's someone else's performance, not the dam's
        if re.match(r'^\s*\d+\s*st[:\s]', line, re.IGNORECASE):
            if debug:
                print(f"  Skipping line {search_start + i} (has starts at beginning): {line[:80]}")
            continue
        
        if debug:
            print(f"  Checking line {search_start + i} for dam name (matches={matches}): {line[:100]}")
        
        # Found a line with the dam name - now look for performance data
        abs_idx = search_start + i
        window = lines[abs_idx:min(abs_idx + 10, len(lines))]
        
        if debug:
            print(f"    Searching in window starting at line {abs_idx}:")
            for wi, wl in enumerate(window[:6]):
                print(f"      {abs_idx + wi}: {wl[:80]}")
        
        dam_best = None
        dam_earn_raw = None
        dam_curr = None
        dam_st = None
        dam_w = None
        dam_p2 = None
        dam_p3 = None
        
        for j, wl in enumerate(window):
            wl_stripped = wl.strip()
            
            # Look for time - skip first line, and skip lines with multiple uppercase words
            if dam_best is None and j > 0:
                time_m = re.search(r'\b(1\.\d{2}[.,]\d|\d{2}[.,]\d)[a-z]{0,2}[lkma]{0,2}\b', wl_stripped, re.IGNORECASE)
                if time_m:
                    # Check if this line has multiple all-caps words (indicates horse names)
                    caps_words = re.findall(r'\b[A-ZÅÄÖÆØÜ]{2,}(?:\s+[A-ZÅÄÖÆØÜ]{2,})+\b', wl_stripped)
                    if not caps_words:
                        raw_time = time_m.group(1).replace(',', '.')
                        dam_best = raw_time
                        if debug:
                            print(f"    Found time in line {abs_idx + j}: {dam_best}")
            
            # Look for earnings with currency
            if dam_earn_raw is None:
                earn_m = re.search(r'([\d\.\s]+)\s*(SEK|EUR|€|NOK|DKK|USD|\$|KR)\b', wl_stripped, re.IGNORECASE)
                if earn_m:
                    dam_earn_raw = _clean_int(earn_m.group(1))
                    dam_curr = earn_m.group(2).upper()
                    if dam_curr == '€':
                        dam_curr = 'EUR'
                    if dam_curr == '$':
                        dam_curr = 'USD'
                    if debug:
                        print(f"    Found earnings in line {abs_idx + j}: {dam_earn_raw} {dam_curr}")
                else:
                    # Fallback: look for number-only line (assume SEK if it's a large number after time)
                    if dam_best is not None and j > 1:  # Only after we found time
                        # Match a line that's just a number (with dots/spaces for thousands)
                        num_only = re.match(r'^\s*([\d\.\s]+)\s*$', wl_stripped)
                        if num_only:
                            potential_earn = _clean_int(num_only.group(1))
                            if potential_earn and potential_earn >= 1000:  # Must be at least 1000 to be earnings
                                dam_earn_raw = potential_earn
                                dam_curr = 'SEK'  # Default to SEK
                                if debug:
                                    print(f"    Found earnings (no currency, assumed SEK) in line {abs_idx + j}: {dam_earn_raw}")
            
            # Look for starts pattern - allow "37st:" format (missing space)
            if dam_st is None:
                starts_m = re.search(r'(\d+)\s*st[:\s]+(\d+)[–-](\d+)[–-](\d+)\b', wl_stripped, re.IGNORECASE)
                if starts_m:
                    dam_st = int(starts_m.group(1))
                    dam_w = int(starts_m.group(2))
                    dam_p2 = int(starts_m.group(3))
                    dam_p3 = int(starts_m.group(4))
                    if debug:
                        print(f"    Found starts in line {abs_idx + j}: {dam_st} st: {dam_w}-{dam_p2}-{dam_p3}")
        
        # If we found at least starts data, return what we have
        if dam_st is not None:
            dam_earn_sek = None
            if dam_earn_raw and dam_curr:
                rate = CURRENCY_RATE.get(dam_curr, 1.0)
                dam_earn_sek = int(round(dam_earn_raw * rate))
            
            if debug:
                print(f"  ✓ Found dam performance:")
                print(f"    Time: {dam_best}, Earnings: {dam_earn_raw} {dam_curr} = {dam_earn_sek} SEK")
                print(f"    Starts: {dam_st} ({dam_w}-{dam_p2}-{dam_p3})")
            
            return dam_best, str(dam_earn_raw) if dam_earn_raw else None, dam_curr, dam_earn_sek, dam_st, dam_w, dam_p2, dam_p3
    
    if debug:
        print(f"  No performance data found for dam (may not have raced)")
    return (None,)*8

def parse_dam_block_scored(dam_name: str, lines: list[str], debug: bool = False):
    if not isinstance(dam_name, str) or not dam_name.strip():
        return (None,)*8

    prefixes = _sig_prefixes(dam_name)
    if not prefixes:
        return (None,)*8
    
    if debug:
        print(f"  Dam name: '{dam_name}' -> Prefixes: {prefixes}")

    def _mostly_caps(s: str) -> bool:
        letters = re.findall(r"[A-Za-zÅÄÖÆØÜÉÈÍÓÚÝÇÑåäöæøüéèíóúýçñ]", s)
        return bool(letters) and sum(ch.isupper() for ch in letters)/len(letters) >= 0.8

    def _is_tagline(s: str) -> bool:
        """Detect promotional/descriptive taglines that should be skipped"""
        if re.search(r"\bf\.?\s*\d{2,4}\b", s, re.IGNORECASE):
            return False
        s_lower = s.lower()
        promotional = ['presenterar', 'stuteri']
        if any(word in s_lower for word in promotional):
            return True
        return False

    # Find all lines with "f. <year>" that match the dam name
    dam_header_candidates = []
    for i, L in enumerate(lines):
        if not re.search(r"\bf\.?\s*\d{2,4}\b", L, re.IGNORECASE):
            continue
        if _is_tagline(L):
            continue
        hits = _line_has_prefixes(L, prefixes)
        if hits > 0:
            is_caps = _mostly_caps(L)
            if debug:
                print(f"  Candidate [{hits} hits, caps={is_caps}] line {i}: {L[:80]}")
            dam_header_candidates.append((hits, int(is_caps), i, L))
    
    if not dam_header_candidates:
        if debug:
            print(f"  NO CANDIDATES FOUND!")
        return (None,)*8
    
    # Sort by: best name match, then CAPS preference, then earliest occurrence
    dam_header_candidates.sort(key=lambda x: (-x[0], -x[1], x[2]))
    
    # Use the best candidate
    idx = dam_header_candidates[0][2]
    matched_line = dam_header_candidates[0][3]
    
    if debug:
        print(f"  SELECTED dam header: line {idx}: {matched_line[:100]}")
    
    # Search for dam's own performance BEFORE the header line
    return parse_dam_own_performance(dam_name, lines, idx, debug=debug)


assert 'df' in globals() and 'pdf_path' in globals(), "Make sure df and pdf_path exist"


# Run on ALL pages without debug - ALWAYS update dam columns
updated = 0
with pdfplumber.open(str(pdf_path)) as pdf:
    for i, row in df.iterrows():
        p = int(row["Page"])
        
        lines = _norm_text(pdf.pages[p-1].extract_text() or "").splitlines()
        best, earn_raw, curr, earn_sek, st, w, p2, p3 = parse_dam_block_scored(row.get("Dam"), lines, debug=False)

        # ALWAYS update, even with None values (to clear incorrect old data)
        df.at[i, "Dam Best Time"]       = best
        df.at[i, "Dam Earnings"]        = earn_raw
        df.at[i, "Dam Currency"]        = curr
        df.at[i, "Dam Earnings (SEK)"]  = earn_sek
        df.at[i, "Dam Starts"]          = st
        df.at[i, "Dam 1st"]             = w
        df.at[i, "Dam 2nd"]             = p2
        df.at[i, "Dam 3rd"]             = p3
        
        # Count as updated if we found at least something
        if any(x is not None for x in [best, earn_raw, curr, earn_sek, st, w, p2, p3]):
            updated += 1

print(f"\nDam performance updated on {updated} rows")



# Save the results
df.to_excel(OUT_V6B_XLSX, index=False)
print(f"\nSaved to {OUT_V6B_XLSX}")



Dam performance updated on 115 rows

Saved to C:\Users\RunarSkjaerpe\SOLVALLA_YS25_FINAL_FIXED_v6b_REENG.xlsx


In [21]:
# 1. Delete empty pages

#df = df[~df["Page"].isin([123, 124])].copy()

# 2. Load excel sheet into DataFrame
df["Number"] = range(1, len(df) + 1)
df_excel = pd.read_excel("C:/Users/RunarSkjaerpe/Solvalla_YS_23.xlsx", sheet_name="Sheet1")

# 3. Merge / join
df_joined = df.merge(df_excel, on="Number", how="left")

# Drop horses that are not sold
df_joined = df_joined.dropna(subset=["Reserve Price"])
df_joined.to_excel(OUT_V6B_XLSX, index=False)